Setup and Raw Data Simulation

#Goal: Create a "dirty" dataset with duplicates, missing values, and outliers.

In [7]:
import pandas as pd
import numpy as np

# 1. Simulating a Complex "Dirty" Dataset
data = {
    'Employee_ID': [101, 102, 103, 104, 105, 101], # 101 is a Duplicate
    'Department': ['IT', 'HR', 'IT', 'Sales', 'HR', 'IT'],
    'Age': [28, 34, 150, 40, 29, 28],              # 150 is an Outlier
    'Salary': [5000, 4500, np.nan, 6000, 4800, 5000], # nan is Missing
    'Bonus_%': [10, 5, 8, 12, np.nan, 10]          # Another missing value
}

df = pd.DataFrame(data)

print("--- Step 1: Raw Data Overview ---")
print(df)
print("\nMissing Values Count:")
print(df.isnull().sum())

--- Step 1: Raw Data Overview ---
   Employee_ID Department  Age  Salary  Bonus_%
0          101         IT   28  5000.0     10.0
1          102         HR   34  4500.0      5.0
2          103         IT  150     NaN      8.0
3          104      Sales   40  6000.0     12.0
4          105         HR   29  4800.0      NaN
5          101         IT   28  5000.0     10.0

Missing Values Count:
Employee_ID    0
Department     0
Age            0
Salary         1
Bonus_%        1
dtype: int64


Cleaning - Duplicates and Outliers

#Goal: Remove redundant data and fix impossible values using the .mask() function.

In [8]:
# A. Remove Duplicates (Industry standard: always do this first)
df = df.drop_duplicates()

# B. Smart Outlier Handling 
# Replace Age > 100 with the median age (more robust than a hardcoded number)
df['Age'] = df['Age'].mask(df['Age'] > 100, df['Age'].median())

print("--- Step 2: After Duplicates & Outlier Fix ---")
print(df)

--- Step 2: After Duplicates & Outlier Fix ---
   Employee_ID Department  Age  Salary  Bonus_%
0          101         IT   28  5000.0     10.0
1          102         HR   34  4500.0      5.0
2          103         IT   34     NaN      8.0
3          104      Sales   40  6000.0     12.0
4          105         HR   29  4800.0      NaN


Advanced Imputation (Filling Missing Data)

#Goal: Use "Grouped Imputation" to fill missing salaries based on Department averages.

In [9]:
# C. Contextual Imputation (Advanced Pro-move)
# If an IT salary is missing, fill it with the mean of the IT department only
df['Salary'] = df.groupby('Department')['Salary'].transform(lambda x: x.fillna(x.mean()))

# D. Simple Imputation
# Fill Bonus with 0 (Assuming if it's NaN, no bonus was earned)
df['Bonus_%'] = df['Bonus_%'].fillna(0)

print("--- Step 3: Fully Imputed Data ---")
print(df)

--- Step 3: Fully Imputed Data ---
   Employee_ID Department  Age  Salary  Bonus_%
0          101         IT   28  5000.0     10.0
1          102         HR   34  4500.0      5.0
2          103         IT   34  5000.0      8.0
3          104      Sales   40  6000.0     12.0
4          105         HR   29  4800.0      0.0


Feature Engineering

#Goal: Create new, high-value columns from existing data (Total Compensation).

In [10]:
# 1. Math across columns
df['Total_Comp'] = df['Salary'] + (df['Salary'] * df['Bonus_%'] / 100)

# 2. Categorical Binning (Creating 'Level' based on Salary)
df['Level'] = df['Salary'].apply(lambda x: 'Senior' if x >= 5000 else 'Junior')

print("--- Step 4: Feature Engineering Complete ---")
print(df[['Employee_ID', 'Department', 'Total_Comp', 'Level']])

--- Step 4: Feature Engineering Complete ---
   Employee_ID Department  Total_Comp   Level
0          101         IT      5500.0  Senior
1          102         HR      4725.0  Junior
2          103         IT      5400.0  Senior
3          104      Sales      6720.0  Senior
4          105         HR      4800.0  Junior


Data Interrogation (Filtering)

#Goal: Extract specific insights from the cleaned data.

In [12]:
# Find all employees in the IT department who are 'Senior' level
it_experts = df[(df['Department'] == 'IT') & (df['Level'] == 'Senior')]

print("--- Final Insight: IT Seniors ---")
print(it_experts)

--- Final Insight: IT Seniors ---
   Employee_ID Department  Age  Salary  Bonus_%  Total_Comp   Level
0          101         IT   28  5000.0     10.0      5500.0  Senior
2          103         IT   34  5000.0      8.0      5400.0  Senior
